# Decision Trees: Visual Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/methods-algorithms/blob/master/notebooks/L04_dt.ipynb)

**Learning Objectives:**
- Understand how decision trees split data using impurity measures
- Visualize decision boundaries and see how tree depth affects complexity
- Compare Gini impurity and entropy as split criteria
- Learn cost-complexity pruning to prevent overfitting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.metrics import accuracy_score

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# ML color palette
ML_PURPLE = '#3333B2'
ML_BLUE = '#0066CC'
ML_ORANGE = '#FF7F0E'
ML_GREEN = '#2CA02C'
ML_RED = '#D62728'

print('Setup complete.')

## 1. Generate Data

In [ ]:
X, y = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, flip_y=0.1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')
print(f'Class distribution: {np.bincount(y)}')

# Decision boundary helper
def plot_decision_boundary(model, X, y, ax, title=''):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    cmap_bg = ListedColormap(['#cce0ff', '#ffe0cc'])
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
    ax.scatter(X[:, 0], X[:, 1], c=[ML_BLUE if yi == 0 else ML_ORANGE for yi in y],
               edgecolors='white', s=40, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.grid(True, alpha=0.3)

# Visual 1: Scatter plot of raw data
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X[:, 0], X[:, 1], c=[ML_BLUE if yi == 0 else ML_ORANGE for yi in y],
           edgecolors='white', s=60, alpha=0.8)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Two-Class Dataset (200 Samples)')
ax.legend(handles=[
    mpatches.Patch(color=ML_BLUE, label='Class 0'),
    mpatches.Patch(color=ML_ORANGE, label='Class 1')
], loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. How a Decision Tree Splits

In [ ]:
# Visual 2: Decision boundary with depth-2 tree
tree_d2 = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_d2.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(10, 6))
plot_decision_boundary(tree_d2, X_train, y_train, ax,
                       title='Decision Tree (max_depth=2) Decision Boundary')
plt.tight_layout()
plt.show()

print(f'Train accuracy: {tree_d2.score(X_train, y_train):.3f}')
print(f'Test accuracy:  {tree_d2.score(X_test, y_test):.3f}')

In [ ]:
# Visual 3: Tree structure visualization
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(tree_d2, filled=True, rounded=True,
          feature_names=['Feature 1', 'Feature 2'],
          class_names=['Class 0', 'Class 1'],
          fontsize=11, ax=ax)
ax.set_title('Tree Structure (max_depth=2)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Effect of Tree Depth

In [ ]:
# Visual 4: 3-panel depth comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

depths = [1, 3, None]
titles = ['Depth=1 (Underfitting)', 'Depth=3 (Balanced)', 'No Limit (Overfitting)']

for ax, depth, title in zip(axes, depths, titles):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_acc = tree.score(X_train, y_train)
    test_acc = tree.score(X_test, y_test)
    plot_decision_boundary(tree, X_train, y_train, ax,
                           title=f'{title}\nTrain: {train_acc:.2f} | Test: {test_acc:.2f}')

plt.suptitle('Effect of Tree Depth on Decision Boundaries', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visual 5: Bias-variance curve
max_depths = range(1, 16)
train_accs = []
test_accs = []

for d in max_depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_accs.append(tree.score(X_train, y_train))
    test_accs.append(tree.score(X_test, y_test))

best_depth = list(max_depths)[np.argmax(test_accs)]
best_test = max(test_accs)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(max_depths), train_accs, 'o-', color=ML_BLUE, linewidth=2, label='Train')
ax.plot(list(max_depths), test_accs, 's-', color=ML_ORANGE, linewidth=2, label='Test')
ax.axvspan(1, 3, alpha=0.1, color='blue', label='Underfitting zone')
ax.axvspan(8, 15, alpha=0.1, color='red', label='Overfitting zone')
ax.scatter([best_depth], [best_test], marker='*', s=300, color=ML_RED, zorder=5)
ax.annotate(f'Best depth={best_depth}\nAcc={best_test:.3f}',
            xy=(best_depth, best_test), xytext=(best_depth + 2, best_test - 0.03),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='black'),
            bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.set_xlabel('Max Depth')
ax.set_ylabel('Accuracy')
ax.set_title('Train vs. Test Accuracy by Tree Depth')
ax.set_xticks(list(max_depths))
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Gini Impurity

In [ ]:
# Visual 6: Gini vs Entropy curves
p = np.linspace(0.001, 0.999, 200)
gini = 2 * p * (1 - p)
entropy = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
entropy_scaled = entropy / 2  # Scale to [0, 0.5] for comparison

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(p, gini, color=ML_BLUE, linewidth=2.5, label='Gini Impurity')
ax.plot(p, entropy_scaled, color=ML_ORANGE, linewidth=2.5, label='Entropy (scaled /2)')
ax.annotate('Pure', xy=(0.02, 0.01), fontsize=12, color=ML_GREEN, fontweight='bold')
ax.annotate('Pure', xy=(0.92, 0.01), fontsize=12, color=ML_GREEN, fontweight='bold')
ax.annotate('Maximum\nImpurity', xy=(0.5, 0.5), fontsize=12, ha='center',
            color=ML_RED, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_xlabel('Proportion of Class 1 (p)')
ax.set_ylabel('Impurity')
ax.set_title('Gini Impurity vs. Entropy')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Regression Trees

In [ ]:
# Visual 7: Regression tree predictions
X_reg = np.sort(np.random.uniform(0, 2 * np.pi, 100)).reshape(-1, 1)
y_reg = np.sin(X_reg.ravel()) + np.random.normal(0, 0.2, 100)
X_plot = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)

reg_d1 = DecisionTreeRegressor(max_depth=1, random_state=42)
reg_d5 = DecisionTreeRegressor(max_depth=5, random_state=42)
reg_d1.fit(X_reg, y_reg)
reg_d5.fit(X_reg, y_reg)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_reg, y_reg, color='gray', alpha=0.5, s=30, label='Data')
ax.plot(X_plot, np.sin(X_plot), 'k--', linewidth=1.5, label='True sin(x)')
ax.plot(X_plot, reg_d1.predict(X_plot), color=ML_BLUE, linewidth=2.5, label='Depth=1')
ax.plot(X_plot, reg_d5.predict(X_plot), color=ML_ORANGE, linewidth=2.5, label='Depth=5')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Regression Trees: Depth=1 vs. Depth=5')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Pruning with Cost-Complexity

In [ ]:
# Visual 8: Pruning curve
clf_full = DecisionTreeClassifier(random_state=42)
clf_full.fit(X_train, y_train)
path = clf_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

train_scores = []
cv_scores = []

for alpha in ccp_alphas:
    tree = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    scores = cross_val_score(tree, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_idx = np.argmax(cv_scores)
best_alpha = ccp_alphas[best_idx]
best_cv = cv_scores[best_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ccp_alphas, train_scores, 'o-', color=ML_BLUE, linewidth=2, markersize=4, label='Train')
ax.plot(ccp_alphas, cv_scores, 's-', color=ML_ORANGE, linewidth=2, markersize=4, label='5-Fold CV')
ax.axvline(x=best_alpha, color=ML_RED, linestyle='--', linewidth=2,
           label=f'Optimal alpha={best_alpha:.4f}')
ax.set_xlabel('Cost-Complexity Parameter (alpha)')
ax.set_ylabel('Accuracy')
ax.set_title('Cost-Complexity Pruning: Finding Optimal Alpha')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Optimal alpha: {best_alpha:.4f}')
print(f'CV accuracy at optimal alpha: {best_cv:.3f}')

## Summary

**Key Takeaways:**
- **Decision trees split data** by choosing the feature and threshold that maximizes impurity reduction (Gini or Entropy)
- **Tree depth controls the bias-variance tradeoff**: shallow trees underfit, deep trees overfit
- **Regression trees** predict the mean of leaf samples and approximate functions with step-like predictions
- **Cost-complexity pruning** finds the right tree size by penalizing complexity -- use cross-validation to select alpha